# Business days in pandas with NYSE, Federal Reserve and US federal holidays

pandas has no NYSE or Federal Reserve calendar. [`pandas-holiday-calendars`](https://pypi.org/project/pandas-holiday-calendars/) adds them (plus US federal, England and Wales, and Euro T2) as ordinary pandas holiday calendars, with the dates embedded in the package. The cells below are the recipes from its README, run end to end.

In [1]:
try:
    import pandas_holiday_calendars
except ImportError:
    %pip install -q pandas-holiday-calendars

## Count NYSE trading days in 2027

`business_day(NYSECalendar)` returns a pandas `CustomBusinessDay`; use it as the `freq` of a date range.

In [2]:
import calendar
import pandas as pd
from pandas_holiday_calendars import NYSECalendar, FederalReserveCalendar, business_day

nyse = business_day(NYSECalendar)
days_2027 = pd.bdate_range("2027-01-01", "2027-12-31", freq=nyse)
print("NYSE trading days in 2027:", len(days_2027))

NYSE trading days in 2027: 251


## A business-day date range for the Federal Reserve calendar

The Fed calendar is a different holiday list, so the same dates can give a different range.

In [3]:
fed = business_day(FederalReserveCalendar)
pd.bdate_range("2026-12-21", "2027-01-05", freq=fed)

DatetimeIndex(['2026-12-21', '2026-12-22', '2026-12-23', '2026-12-24',
               '2026-12-28', '2026-12-29', '2026-12-30', '2026-12-31',
               '2027-01-04', '2027-01-05'],
              dtype='datetime64[us]', freq='C')

## Add N business days

In [4]:
print(pd.Timestamp("2026-11-24") + 3 * nyse)   # Thanksgiving is 2026-11-26
print(pd.Timestamp("2026-07-06") - 1 * nyse)   # 2026-07-03 is Independence Day (observed)

2026-11-30 00:00:00
2026-07-02 00:00:00


## Count business days with numpy

`busdaycalendar` returns a `numpy.busdaycalendar`. `busday_count` counts the start date and excludes the end date.

In [5]:
import numpy as np
from pandas_holiday_calendars import busdaycalendar

bc = busdaycalendar("NYSECalendar")
print(np.busday_count("2026-12-01", "2027-01-01", busdaycal=bc))
print(np.is_busday("2026-12-25", busdaycal=bc))
print(np.busday_offset("2026-12-24", 1, busdaycal=bc))

22
False
2026-12-28


## Guard against years outside the data

pandas treats a year with no holiday data as a year without holidays and does not warn. `check_range` raises instead. The NYSE and Fed lists end in 2050.

In [6]:
from pandas_holiday_calendars import check_range

try:
    check_range("NYSECalendar", "2051-01-01", "2051-12-31")
except ValueError as err:
    print("ValueError:", err)

ValueError: NYSECalendar covers 2000-2050; 2051-01-01..2051-12-31 is outside


## NYSE trading days per month, 2027

In [7]:
per_month = days_2027.to_series().groupby(days_2027.month).size()
table = pd.DataFrame({
    "month": [calendar.month_abbr[m] for m in per_month.index],
    "trading_days": per_month.values,
})
print("Total:", table["trading_days"].sum())
table

Total: 251


,month,trading_days
0,Jan,19
1,Feb,19
2,Mar,22
3,Apr,22
4,May,20
5,Jun,21
6,Jul,21
7,Aug,22
8,Sep,21
9,Oct,21


## Next

- Trading days for every year, free: https://plainfoldhq.github.io/trading-days-per-year/
- Other years or countries: see the "Need other years or countries?" section of the [README](https://github.com/plainfoldhq/pandas-holiday-calendars#need-other-years-or-countries).

Holiday calendars are republished by authorities; check the official source where a wrong date has a cost.

Plainfold is an AI-run business; this notebook was written and tested by AI agents.